In [1]:
import pandas as pd

df = pd.read_csv('data/P1_Mestre_5epocas.csv')

df.head()

,Div,Date,Time,HomeTeam,AwayTeam,FTHG,FTAG,FTR,HTHG,HTAG,...,BMGMCA,BVCH,BVCD,BVCA,CLCH,CLCD,CLCA,LBCH,LBCD,LBCA
0,P1,2021-08-06,20:15,Sp Lisbon,Vizela,3,0,H,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,P1,2021-08-07,12:45,Arouca,Estoril,0,2,A,0,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,P1,2021-08-07,18:00,Moreirense,Benfica,1,2,A,1,2,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,P1,2021-08-07,20:30,Maritimo,Sp Braga,0,2,A,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,P1,2021-08-08,15:30,Guimaraes,Portimonense,0,1,A,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [2]:

print("Número de colunas originais:", len(df.columns))

colunas_importantes = ['Date', 'HomeTeam', 'AwayTeam', 'FTHG', 'FTAG', 'FTR', 'HST', 'AST']


df_limpo = df[colunas_importantes].copy()


df_limpo.head()

Número de colunas originais: 161


,Date,HomeTeam,AwayTeam,FTHG,FTAG,FTR,HST,AST
0,2021-08-06,Sp Lisbon,Vizela,3,0,H,5,1
1,2021-08-07,Arouca,Estoril,0,2,A,3,6
2,2021-08-07,Moreirense,Benfica,1,2,A,3,6
3,2021-08-07,Maritimo,Sp Braga,0,2,A,0,5
4,2021-08-08,Guimaraes,Portimonense,0,1,A,3,1


In [3]:
import pandas as pd 


df_limpo['Date'] = pd.to_datetime(df_limpo['Date'], format='mixed')
df_limpo = df_limpo.sort_values(by='Date')


def calcula_pontos_casa(resultado):
    if resultado == 'H': 
        return 3
    elif resultado == 'D': 
        return 1
    else: 
        return 0

def calcula_pontos_fora(resultado):
    if resultado == 'A': 
        return 3
    elif resultado == 'D': 
        return 1
    else: 
        return 0


df_limpo['HomePoints'] = df_limpo['FTR'].apply(calcula_pontos_casa)
df_limpo['AwayPoints'] = df_limpo['FTR'].apply(calcula_pontos_fora)


df_limpo.head()

,Date,HomeTeam,AwayTeam,FTHG,FTAG,FTR,HST,AST,HomePoints,AwayPoints
0,2021-08-06,Sp Lisbon,Vizela,3,0,H,5,1,3,0
1,2021-08-07,Arouca,Estoril,0,2,A,3,6,0,3
2,2021-08-07,Moreirense,Benfica,1,2,A,3,6,0,3
3,2021-08-07,Maritimo,Sp Braga,0,2,A,0,5,0,3
4,2021-08-08,Guimaraes,Portimonense,0,1,A,3,1,0,3


In [4]:

df_limpo['HomeForm'] = 0
df_limpo['AwayForm'] = 0


pontos_totais = {}


for index, row in df_limpo.iterrows():
    casa = row['HomeTeam']
    fora = row['AwayTeam']
    
    
    if casa not in pontos_totais:
        pontos_totais[casa] = 0
    if fora not in pontos_totais:
        pontos_totais[fora] = 0
        
    
    df_limpo.at[index, 'HomeForm'] = pontos_totais[casa]
    df_limpo.at[index, 'AwayForm'] = pontos_totais[fora]
    
    
    pontos_totais[casa] += row['HomePoints']
    pontos_totais[fora] += row['AwayPoints']


df_limpo.head(15)

,Date,HomeTeam,AwayTeam,FTHG,FTAG,FTR,HST,AST,HomePoints,AwayPoints,HomeForm,AwayForm
0,2021-08-06,Sp Lisbon,Vizela,3,0,H,5,1,3,0,0,0
1,2021-08-07,Arouca,Estoril,0,2,A,3,6,0,3,0,0
2,2021-08-07,Moreirense,Benfica,1,2,A,3,6,0,3,0,0
3,2021-08-07,Maritimo,Sp Braga,0,2,A,0,5,0,3,0,0
4,2021-08-08,Guimaraes,Portimonense,0,1,A,3,1,0,3,0,0
5,2021-08-08,Tondela,Santa Clara,3,0,H,5,5,3,0,0,0
6,2021-08-08,Porto,Belenenses,2,0,H,7,3,3,0,0,0
7,2021-08-08,Pacos Ferreira,Famalicao,2,0,H,3,2,3,0,0,0
8,2021-08-09,Gil Vicente,Boavista,3,0,H,6,5,3,0,0,0
9,2021-08-13,Estoril,Guimaraes,0,0,D,1,2,1,1,3,0


In [5]:
# 1. Definir o nosso Alvo (O que queremos prever)
y = df_limpo['FTR']

# 2. Definir as nossas Variáveis (O que vamos usar para prever)
# Vamos apagar a Data e as colunas que são "batota" (Data Leakage)
X = df_limpo.drop(columns=['Date', 'FTHG', 'FTAG', 'FTR', 'HomePoints', 'AwayPoints'])

# 3. Converter o texto (Nomes das Equipas) em números (0 e 1)
X = pd.get_dummies(X, columns=['HomeTeam', 'AwayTeam'], dtype=int)

# Vamos ver a nossa tabela X final, pronta para ir para o forno!
X.head()

,HST,AST,HomeForm,AwayForm,HomeTeam_AVS,HomeTeam_Alverca,HomeTeam_Arouca,HomeTeam_Belenenses,HomeTeam_Benfica,HomeTeam_Boavista,...,AwayTeam_Nacional,AwayTeam_Pacos Ferreira,AwayTeam_Portimonense,AwayTeam_Porto,AwayTeam_Rio Ave,AwayTeam_Santa Clara,AwayTeam_Sp Braga,AwayTeam_Sp Lisbon,AwayTeam_Tondela,AwayTeam_Vizela
0,5,1,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
1,3,6,0,0,0,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,3,6,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,5,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,1,0,0,0
4,3,1,0,0,0,0,0,0,0,0,...,0,0,1,0,0,0,0,0,0,0


In [6]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

X_train, X_test, y_train, y_test = train_test_split (X, y, test_size=0.2, random_state=42)

print(f"Jogos para o modelo estudar: {len(X_train)}")
print(f"Jogos para o modelo testar: {len(X_test)}")

modelo=RandomForestClassifier(n_estimators=200, random_state=42)

modelo.fit(X_train, y_train)

previsoes= modelo.predict(X_test)

precisao = accuracy_score(y_test, previsoes)
print(f"\n--->Precisão do modelo: {precisao * 100:.2f}%")

Jogos para o modelo estudar: 1216
Jogos para o modelo testar: 305

--->Precisão do modelo: 57.05%


In [7]:
from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score

# 1. O Tradutor: Converter 'H', 'D', 'A' em 0, 1, 2 para o XGBoost não se queixar
tradutor = LabelEncoder()
y_train_numeros = tradutor.fit_transform(y_train)
y_test_numeros = tradutor.transform(y_test)

# 2. Criar o Monstro (Vamos usar as mesmas 200 "árvores" para ser justo)
# A learning_rate (taxa de aprendizagem) impede o modelo de ser demasiado confiante e errar
modelo_xgb = XGBClassifier(
    n_estimators=200, 
    learning_rate=0.05, 
    max_depth=3, # Árvores baixinhas para evitar que ele decore o ficheiro
    random_state=42
)

# 3. Treinar a máquina
print("A treinar o XGBoost. Aguenta coração...")
modelo_xgb.fit(X_train, y_train_numeros)

# 4. Fazer o teste final e ver o resultado
previsoes_xgb = modelo_xgb.predict(X_test)
precisao_xgb = accuracy_score(y_test_numeros, previsoes_xgb)

print(f"\n🚀 Precisão Final do XGBoost: {precisao_xgb * 100:.2f}%")

A treinar o XGBoost. Aguenta coração...

🚀 Precisão Final do XGBoost: 56.39%


In [8]:
import joblib

# 1. Agarrar no nosso modelo vencedor das 200 árvores (da Célula 6)
modelo_final = modelo 

# 2. Exportar o modelo para um ficheiro chamado .pkl (Pickle)
joblib.dump(modelo_final, 'modelo_primeira_liga.pkl')

# 3. Guardar a "lista de ingredientes" (as colunas do nosso X)
colunas_do_modelo = list(X.columns)
joblib.dump(colunas_do_modelo, 'colunas_modelo.pkl')

print("🧠 Cérebro do modelo e colunas guardados com sucesso no teu Mac!")

🧠 Cérebro do modelo e colunas guardados com sucesso no teu Mac!


In [9]:
import joblib
import pandas as pd

# 1. Acordar o "Cérebro" (Carregar ficheiros)
modelo = joblib.load('modelo_primeira_liga.pkl')
colunas = joblib.load('colunas_modelo.pkl')

# 2. Criar uma tabela em branco com o formato exato que o modelo exige
jogo_novo = pd.DataFrame(0, index=[0], columns=colunas)

# 3. Preencher os dados do nosso jogo fictício
# IMPORTANTE: Os nomes das equipas têm de estar escritos exatamente como no CSV
jogo_novo.at[0, 'HomeForm'] = 15  # Pontos do Porto à entrada para o jogo
jogo_novo.at[0, 'AwayForm'] = 14  # Pontos do Benfica à entrada para o jogo
jogo_novo.at[0, 'HomeTeam_Porto'] = 1
jogo_novo.at[0, 'AwayTeam_Benfica'] = 1

# 4. Pedir à máquina para prever!
probabilidades = modelo.predict_proba(jogo_novo)[0]

print("⚽ PREVISÃO DO GRANDE CLÁSSICO ⚽")
print(f"Vitória do Porto (Casa): {probabilidades[2] * 100:.1f}%")
print(f"Empate:                  {probabilidades[1] * 100:.1f}%")
print(f"Vitória do Benfica (Fora): {probabilidades[0] * 100:.1f}%")

⚽ PREVISÃO DO GRANDE CLÁSSICO ⚽
Vitória do Porto (Casa): 39.0%
Empate:                  36.0%
Vitória do Benfica (Fora): 25.0%


In [10]:
# 1. Definir o tamanho da nossa janela
N_JOGOS = 6

# 2. Criar as novas colunas a zeros
df_limpo['HomeGolosMarcados_Ultimos6'] = 0.0
df_limpo['HomeGolosSofridos_Ultimos6'] = 0.0
df_limpo['HomePontos_Ultimos6'] = 0.0
df_limpo['HomeRematesBaliza_Ultimos6'] = 0.0 # A nossa nova arma (xG)

df_limpo['AwayGolosMarcados_Ultimos6'] = 0.0
df_limpo['AwayGolosSofridos_Ultimos6'] = 0.0
df_limpo['AwayPontos_Ultimos6'] = 0.0
df_limpo['AwayRematesBaliza_Ultimos6'] = 0.0

# 3. Dicionário complexo para guardar o histórico temporário
historico = {}

for index, row in df_limpo.iterrows():
    casa = row['HomeTeam']
    fora = row['AwayTeam']
    
    # Criar listas vazias se a equipa for nova
    if casa not in historico:
        historico[casa] = {'marcados': [], 'sofridos': [], 'pontos': [], 'remates': []}
    if fora not in historico:
        historico[fora] = {'marcados': [], 'sofridos': [], 'pontos': [], 'remates': []}
        
    # --- PARTE A: LER A MEMÓRIA (ANTES DO JOGO) ---
    if len(historico[casa]['pontos']) > 0:
        df_limpo.at[index, 'HomeGolosMarcados_Ultimos6'] = sum(historico[casa]['marcados']) / len(historico[casa]['marcados'])
        df_limpo.at[index, 'HomeGolosSofridos_Ultimos6'] = sum(historico[casa]['sofridos']) / len(historico[casa]['sofridos'])
        df_limpo.at[index, 'HomePontos_Ultimos6'] = sum(historico[casa]['pontos'])
        df_limpo.at[index, 'HomeRematesBaliza_Ultimos6'] = sum(historico[casa]['remates']) / len(historico[casa]['remates'])
        
    if len(historico[fora]['pontos']) > 0:
        df_limpo.at[index, 'AwayGolosMarcados_Ultimos6'] = sum(historico[fora]['marcados']) / len(historico[fora]['marcados'])
        df_limpo.at[index, 'AwayGolosSofridos_Ultimos6'] = sum(historico[fora]['sofridos']) / len(historico[fora]['sofridos'])
        df_limpo.at[index, 'AwayPontos_Ultimos6'] = sum(historico[fora]['pontos'])
        df_limpo.at[index, 'AwayRematesBaliza_Ultimos6'] = sum(historico[fora]['remates']) / len(historico[fora]['remates'])
        
    # --- PARTE B: ATUALIZAR A MEMÓRIA (DEPOIS DO JOGO) ---
    historico[casa]['marcados'].append(row['FTHG'])
    historico[casa]['sofridos'].append(row['FTAG'])
    historico[casa]['pontos'].append(row['HomePoints'])
    # Se a coluna HST falhar por algum motivo (dados antigos), guardamos 0
    historico[casa]['remates'].append(row.get('HST', 0) if pd.notna(row.get('HST', 0)) else 0)
    
    historico[fora]['marcados'].append(row['FTAG'])
    historico[fora]['sofridos'].append(row['FTHG'])
    historico[fora]['pontos'].append(row['AwayPoints'])
    historico[fora]['remates'].append(row.get('AST', 0) if pd.notna(row.get('AST', 0)) else 0)
    
    # Manter apenas os últimos 6
    for equipa in [casa, fora]:
        historico[equipa]['marcados'] = historico[equipa]['marcados'][-N_JOGOS:]
        historico[equipa]['sofridos'] = historico[equipa]['sofridos'][-N_JOGOS:]
        historico[equipa]['pontos']   = historico[equipa]['pontos'][-N_JOGOS:]
        historico[equipa]['remates']  = historico[equipa]['remates'][-N_JOGOS:]

df_limpo.tail()

,Date,HomeTeam,AwayTeam,FTHG,FTAG,FTR,HST,AST,HomePoints,AwayPoints,HomeForm,AwayForm,HomeGolosMarcados_Ultimos6,HomeGolosSofridos_Ultimos6,HomePontos_Ultimos6,HomeRematesBaliza_Ultimos6,AwayGolosMarcados_Ultimos6,AwayGolosSofridos_Ultimos6,AwayPontos_Ultimos6,AwayRematesBaliza_Ultimos6
1519,2026-05-11,Gil Vicente,Arouca,1,3,A,5,6,0,3,208,205,1.166667,0.833333,8.0,4.000000,1.333333,1.166667,10.0,3.000000
1515,2026-05-11,Guimaraes,Casa Pia,0,1,A,2,5,0,3,260,150,1.666667,1.500000,10.0,4.000000,0.500000,1.333333,2.0,2.666667
1514,2026-05-11,Rio Ave,Sp Lisbon,1,4,A,3,5,0,3,150,407,1.166667,1.166667,8.0,5.000000,2.333333,1.333333,11.0,6.500000
1516,2026-05-11,Santa Clara,Nacional,2,0,H,2,1,3,0,152,65,1.166667,1.500000,8.0,3.166667,1.000000,0.833333,9.0,5.333333
1520,2026-05-11,Tondela,Moreirense,2,0,H,3,2,3,0,53,166,0.833333,2.166667,5.0,3.500000,1.000000,1.500000,7.0,3.000000


In [11]:
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score

# 1. Preparar o novo X e y (Lembrar de apagar a "batota")
y_v2 = df_limpo['FTR']
X_v2 = df_limpo.drop(columns=['Date', 'FTHG', 'FTAG', 'FTR', 'HomePoints', 'AwayPoints'])
X_v2 = pd.get_dummies(X_v2, columns=['HomeTeam', 'AwayTeam'], dtype=int)

# 2. Cortar os dados: 80% para Treino, 20% para Teste (usando o mesmo random_state=42)
X_train_v2, X_test_v2, y_train_v2, y_test_v2 = train_test_split(X_v2, y_v2, test_size=0.2, random_state=42)

# 3. Tradutor do XGBoost (Converter 'H', 'D', 'A' em 0, 1, 2)
tradutor = LabelEncoder()
y_train_num = tradutor.fit_transform(y_train_v2)
y_test_num = tradutor.transform(y_test_v2)

# 4. Acordar o Monstro! 
modelo_xgb_v2 = XGBClassifier(
    n_estimators=200, 
    learning_rate=0.05, 
    max_depth=3, 
    random_state=42
)

# 5. Treinar com as novas métricas e Testar
print("A treinar o XGBoost com as novas métricas táticas...")
modelo_xgb_v2.fit(X_train_v2, y_train_num)

previsoes_xgb_v2 = modelo_xgb_v2.predict(X_test_v2)
precisao_xgb_v2 = accuracy_score(y_test_num, previsoes_xgb_v2)

print(f"\n🚀 Nova Precisão Final do XGBoost: {precisao_xgb_v2 * 100:.2f}%")

A treinar o XGBoost com as novas métricas táticas...

🚀 Nova Precisão Final do XGBoost: 58.03%


In [12]:
from sklearn.ensemble import RandomForestClassifier

# 1. Acordar o nosso Jipe (com as fiáveis 200 árvores)
modelo_rf_v2 = RandomForestClassifier(n_estimators=200, random_state=42)

# 2. Treinar com as novas variáveis táticas (X_train_v2 e y_train_v2 já estão criados!)
print("A treinar o Campeão (Random Forest) com as novas métricas...")
modelo_rf_v2.fit(X_train_v2, y_train_v2)

# 3. Fazer o teste final
previsoes_rf_v2 = modelo_rf_v2.predict(X_test_v2)
precisao_rf_v2 = accuracy_score(y_test_v2, previsoes_rf_v2)

print(f"\n🏆 Nova Precisão do Random Forest v2.0: {precisao_rf_v2 * 100:.2f}%")

A treinar o Campeão (Random Forest) com as novas métricas...

🏆 Nova Precisão do Random Forest v2.0: 56.72%


In [13]:
# 1. Descobrir quais são os jogos iniciais
# Vamos contar quantos jogos em casa a equipa já fez até àquela linha
df_limpo['Jogos_Casa_Realizados'] = df_limpo.groupby('HomeTeam').cumcount()

# 2. Atribuir os Pesos
df_limpo['Peso_Jogo'] = 1.0 # Por defeito, todos os jogos valem 100% de importância
# Se a equipa tiver menos de 6 jogos realizados, o jogo passa a valer apenas 30% na cabeça do modelo
df_limpo.loc[df_limpo['Jogos_Casa_Realizados'] < 6, 'Peso_Jogo'] = 0.3

# Extrair os pesos para uma variável separada
pesos_totais = df_limpo['Peso_Jogo']

# 3. Fazer o corte dos dados (mas agora cortamos também os pesos para bater tudo certo!)
X_train_v3, X_test_v3, y_train_v3, y_test_v3, pesos_train, pesos_test = train_test_split(
    X_v2, y_v2, pesos_totais, test_size=0.2, random_state=42
)

# 4. Acordar o Campeão e Treinar COM OS PESOS
modelo_rf_v3 = RandomForestClassifier(n_estimators=200, random_state=42)

print("A treinar o Random Forest (com as médias de golos e pesos corrigidos)...")
# A MAGIA ACONTECE AQUI (sample_weight):
modelo_rf_v3.fit(X_train_v3, y_train_v3, sample_weight=pesos_train)

# 5. Fazer o teste final
previsoes_rf_v3 = modelo_rf_v3.predict(X_test_v3)
precisao_rf_v3 = accuracy_score(y_test_v3, previsoes_rf_v3)

print(f"\n🧠 Precisão Final com Pesos de Amostra: {precisao_rf_v3 * 100:.2f}%")

A treinar o Random Forest (com as médias de golos e pesos corrigidos)...

🧠 Precisão Final com Pesos de Amostra: 55.74%


In [14]:
# 1. Criar a coluna e definir o valor "Neutro" (1 ponto) para equipas que nunca se defrontaram
df_limpo['H2H_Pontos_Casa'] = 1.0 

# 2. Dicionário para guardar o histórico exclusivo de cada par de equipas
# A chave vai ser uma dupla ordenada: ex: ('Arouca', 'Porto')
h2h_memoria = {}

for index, row in df_limpo.iterrows():
    casa = row['HomeTeam']
    fora = row['AwayTeam']
    
    # Criar uma "etiqueta" única para este par, independentemente de quem joga em casa
    confronto = tuple(sorted([casa, fora]))
    
    if confronto not in h2h_memoria:
        h2h_memoria[confronto] = []
        
    # --- PARTE A: LER A MEMÓRIA (ANTES DO JOGO) ---
    # Se já jogaram antes, calcular a média de pontos que a equipa 'casa' conseguiu ganhar ao 'fora'
    if len(h2h_memoria[confronto]) > 0:
        pontos_conquistados = 0
        for vencedor_passado in h2h_memoria[confronto]:
            if vencedor_passado == casa:
                pontos_conquistados += 3
            elif vencedor_passado == 'Empate':
                pontos_conquistados += 1
                
        # A média de pontos por jogo neste confronto (varia entre 0.0 e 3.0)
        df_limpo.at[index, 'H2H_Pontos_Casa'] = pontos_conquistados / len(h2h_memoria[confronto])
        
    # --- PARTE B: ATUALIZAR A MEMÓRIA (DEPOIS DO JOGO) ---
    if row['FTR'] == 'H':
        vencedor_hoje = casa
    elif row['FTR'] == 'A':
        vencedor_hoje = fora
    else:
        vencedor_hoje = 'Empate'
        
    h2h_memoria[confronto].append(vencedor_hoje)
    
    # Manter apenas os últimos 6 confrontos diretos na memória (para captar a rivalidade recente)
    h2h_memoria[confronto] = h2h_memoria[confronto][-6:]

# Vamos espreitar alguns jogos grandes para ver se a rivalidade está lá!
df_limpo[['Date', 'HomeTeam', 'AwayTeam', 'H2H_Pontos_Casa']].tail(15)

,Date,HomeTeam,AwayTeam,H2H_Pontos_Casa
1506,2026-05-02,Famalicao,Benfica,1.000000
1507,2026-05-02,Porto,Alverca,3.000000
1508,2026-05-03,Casa Pia,Tondela,3.000000
1509,2026-05-03,Sp Braga,Estoril,2.166667
1510,2026-05-03,Rio Ave,Gil Vicente,1.666667
1511,2026-05-04,Sp Lisbon,Guimaraes,2.166667
1512,2026-05-10,AVS,Porto,0.000000
1513,2026-05-10,Alverca,Estoril,0.000000
1518,2026-05-11,Benfica,Sp Braga,1.833333
1517,2026-05-11,Estrela,Famalicao,1.600000


In [15]:
# 1. Criar as novas colunas
df_limpo['HomeDiasDescanso'] = 0
df_limpo['AwayDiasDescanso'] = 0

# 2. Dicionário para memorizar o calendário de cada equipa
ultima_data_jogo = {}

# O nosso teto máximo: acima de 14 dias, a equipa está 100% fresca
LIMITE_DESCANSO = 14 

for index, row in df_limpo.iterrows():
    casa = row['HomeTeam']
    fora = row['AwayTeam']
    data_atual = row['Date']
    
    # --- CALCULAR DESCANSO DA CASA ---
    if casa in ultima_data_jogo:
        dias_casa = (data_atual - ultima_data_jogo[casa]).days
        df_limpo.at[index, 'HomeDiasDescanso'] = min(dias_casa, LIMITE_DESCANSO)
    else:
        # Se for o primeiro jogo de sempre da equipa no ficheiro, assumimos que estão frescos
        df_limpo.at[index, 'HomeDiasDescanso'] = LIMITE_DESCANSO 
        
    # --- CALCULAR DESCANSO DE FORA ---
    if fora in ultima_data_jogo:
        dias_fora = (data_atual - ultima_data_jogo[fora]).days
        df_limpo.at[index, 'AwayDiasDescanso'] = min(dias_fora, LIMITE_DESCANSO)
    else:
        df_limpo.at[index, 'AwayDiasDescanso'] = LIMITE_DESCANSO
        
    # --- ATUALIZAR A MEMÓRIA ---
    # Depois do jogo de hoje acabar, esta passa a ser a última data em que jogaram
    ultima_data_jogo[casa] = data_atual
    ultima_data_jogo[fora] = data_atual

# Vamos espreitar para ver o desgaste biológico a funcionar!
df_limpo[['Date', 'HomeTeam', 'AwayTeam', 'HomeDiasDescanso', 'AwayDiasDescanso']].tail(15)

,Date,HomeTeam,AwayTeam,HomeDiasDescanso,AwayDiasDescanso
1506,2026-05-02,Famalicao,Benfica,6,7
1507,2026-05-02,Porto,Alverca,6,8
1508,2026-05-03,Casa Pia,Tondela,6,4
1509,2026-05-03,Sp Braga,Estoril,7,7
1510,2026-05-03,Rio Ave,Gil Vicente,8,6
1511,2026-05-04,Sp Lisbon,Guimaraes,5,9
1512,2026-05-10,AVS,Porto,8,8
1513,2026-05-10,Alverca,Estoril,8,7
1518,2026-05-11,Benfica,Sp Braga,9,8
1517,2026-05-11,Estrela,Famalicao,9,9


In [16]:
# 1. Criar as novas colunas a zeros
df_limpo['Home_Streak'] = 0
df_limpo['Away_Streak'] = 0

# 2. Dicionário de memória para guardar o "estado de espírito" atual de cada equipa
memoria_streak = {}

for index, row in df_limpo.iterrows():
    casa = row['HomeTeam']
    fora = row['AwayTeam']
    
    # Iniciar a equipa a zero se for a primeira vez que a vemos
    if casa not in memoria_streak:
        memoria_streak[casa] = 0
    if fora not in memoria_streak:
        memoria_streak[fora] = 0
        
    # --- PARTE A: LER A MEMÓRIA (O ESTADO PSICOLÓGICO ANTES DO JOGO) ---
    df_limpo.at[index, 'Home_Streak'] = memoria_streak[casa]
    df_limpo.at[index, 'Away_Streak'] = memoria_streak[fora]
    
    # --- PARTE B: ATUALIZAR A MEMÓRIA (O IMPACTO DO RESULTADO DE HOJE) ---
    
    # Atualizar o orgulho da equipa da Casa
    if row['FTR'] == 'H': # Casa Ganhou
        memoria_streak[casa] = memoria_streak[casa] + 1 if memoria_streak[casa] > 0 else 1
    elif row['FTR'] == 'A': # Casa Perdeu
        memoria_streak[casa] = memoria_streak[casa] - 1 if memoria_streak[casa] < 0 else -1
    else: # Empate
        memoria_streak[casa] = 0
        
    # Atualizar o orgulho da equipa de Fora
    if row['FTR'] == 'A': # Fora Ganhou
        memoria_streak[fora] = memoria_streak[fora] + 1 if memoria_streak[fora] > 0 else 1
    elif row['FTR'] == 'H': # Fora Perdeu
        memoria_streak[fora] = memoria_streak[fora] - 1 if memoria_streak[fora] < 0 else -1
    else: # Empate
        memoria_streak[fora] = 0

# Vamos ver o estado de espírito das equipas nas últimas 15 jornadas
df_limpo[['Date', 'HomeTeam', 'AwayTeam', 'FTR', 'Home_Streak', 'Away_Streak']].tail(15)

,Date,HomeTeam,AwayTeam,FTR,Home_Streak,Away_Streak
1506,2026-05-02,Famalicao,Benfica,D,1,3
1507,2026-05-02,Porto,Alverca,H,3,1
1508,2026-05-03,Casa Pia,Tondela,A,-2,0
1509,2026-05-03,Sp Braga,Estoril,D,-1,-5
1510,2026-05-03,Rio Ave,Gil Vicente,D,-1,1
1511,2026-05-04,Sp Lisbon,Guimaraes,H,0,2
1512,2026-05-10,AVS,Porto,H,1,4
1513,2026-05-10,Alverca,Estoril,D,-1,0
1518,2026-05-11,Benfica,Sp Braga,D,0,0
1517,2026-05-11,Estrela,Famalicao,D,-5,0


In [17]:
# 1. Definir o valor base de todas as equipas e a velocidade de mudança (K-factor)
ELO_INICIAL = 1500.0
K_FACTOR = 30 # O número máximo de pontos Elo que se pode ganhar ou perder num só jogo

# 2. Criar as colunas a zeros
df_limpo['Home_Elo'] = 0.0
df_limpo['Away_Elo'] = 0.0

# 3. Dicionário para guardar o Elo atualizado de cada equipa
elo_equipas = {}

def calcular_elo(elo_a, elo_b, resultado_a):
    """
    Fórmula oficial do Elo:
    resultado_a: 1 (Vitória), 0.5 (Empate), 0 (Derrota)
    """
    # Probabilidade esperada de a equipa A ganhar
    esperado_a = 1 / (1 + 10 ** ((elo_b - elo_a) / 400))
    # Novo Elo
    novo_elo_a = elo_a + K_FACTOR * (resultado_a - esperado_a)
    return novo_elo_a

for index, row in df_limpo.iterrows():
    casa = row['HomeTeam']
    fora = row['AwayTeam']
    
    # Se a equipa for nova no sistema, entra com 1500 pontos
    if casa not in elo_equipas:
        elo_equipas[casa] = ELO_INICIAL
    if fora not in elo_equipas:
        elo_equipas[fora] = ELO_INICIAL
        
    # --- PARTE A: LER O ELO (O QUE O MODELO VAI USAR PARA PREVER O JOGO) ---
    df_limpo.at[index, 'Home_Elo'] = elo_equipas[casa]
    df_limpo.at[index, 'Away_Elo'] = elo_equipas[fora]
    
    # --- PARTE B: ATUALIZAR O ELO (DEPOIS DE SABERMOS O RESULTADO) ---
    if row['FTR'] == 'H':
        resultado_casa = 1.0
        resultado_fora = 0.0
    elif row['FTR'] == 'A':
        resultado_casa = 0.0
        resultado_fora = 1.0
    else:
        resultado_casa = 0.5
        resultado_fora = 0.5
        
    elo_atual_casa = elo_equipas[casa]
    elo_atual_fora = elo_equipas[fora]
    
    # Calcular e atualizar na memória para o próximo jogo!
    elo_equipas[casa] = calcular_elo(elo_atual_casa, elo_atual_fora, resultado_casa)
    elo_equipas[fora] = calcular_elo(elo_atual_fora, elo_atual_casa, resultado_fora)

# Vamos ver como está o "Poder" (Elo) de cada equipa na última jornada
df_limpo[['Date', 'HomeTeam', 'AwayTeam', 'Home_Elo', 'Away_Elo']].tail(15)

,Date,HomeTeam,AwayTeam,Home_Elo,Away_Elo
1506,2026-05-02,Famalicao,Benfica,1600.850315,1836.233723
1507,2026-05-02,Porto,Alverca,1826.948816,1510.285566
1508,2026-05-03,Casa Pia,Tondela,1439.834376,1387.412245
1509,2026-05-03,Sp Braga,Estoril,1668.746699,1486.511073
1510,2026-05-03,Rio Ave,Gil Vicente,1473.182060,1530.945511
1511,2026-05-04,Sp Lisbon,Guimaraes,1817.904534,1548.054234
1512,2026-05-10,AVS,Porto,1381.801916,1831.121546
1513,2026-05-10,Alverca,Estoril,1506.112835,1493.728746
1518,2026-05-11,Benfica,Sp Braga,1827.385466,1661.529026
1517,2026-05-11,Estrela,Famalicao,1395.135651,1609.698573


In [18]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score
import pandas as pd

print("A preparar os dados e a iniciar os testes no Laboratório...\n")

# 1. Isolar a variável dos Pesos (Sample Weights)
pesos_totais = df_limpo['Peso_Jogo']

# 2. Definir o Alvo (y) e as Features (X)
y = df_limpo['FTR']

# CORRECÇÃO RADICAL: Deitar fora os nomes das equipas para limpar o ruído fantasma!
X = df_limpo.drop(columns=[
    'Date', 'FTHG', 'FTAG', 'FTR', 
    'HomePoints', 'AwayPoints', 
    'HomeTeam', 'AwayTeam', # <--- ADICIONAR ESTAS DUAS AQUI!
    'Jogos_Casa_Realizados', 'Peso_Jogo', 'HST', 'AST'
])

# 4. Dividir em Treino e Teste (80/20) levando os pesos atrás
X_train, X_test, y_train, y_test, pesos_train, pesos_test = train_test_split(
    X, y, pesos_totais, test_size=0.2, random_state=42
)

# 5. Tradutor de texto para números para o XGBoost funcionar
tradutor = LabelEncoder()
y_train_num = tradutor.fit_transform(y_train)
y_test_num = tradutor.transform(y_test)

# --- TREINO E TESTE: RANDOM FOREST (O NOSSO CAMPEÃO ATUAL) ---
rf = RandomForestClassifier(n_estimators=200, random_state=42)
rf.fit(X_train, y_train, sample_weight=pesos_train)
acc_rf = accuracy_score(y_test, rf.predict(X_test))

# --- TREINO E TESTE: XGBOOST (AGORA COM O MOTOR AFINADO PELO GRID SEARCH) ---
xgb = XGBClassifier(
    n_estimators=100,       # Atualizado
    learning_rate=0.05,     # Atualizado
    max_depth=2,            # Atualizado
    subsample=0.8,          # Atualizado
    random_state=42
)
xgb.fit(X_train, y_train_num, sample_weight=pesos_train)
acc_xgb = accuracy_score(y_test_num, xgb.predict(X_test))

# --- RESULTADOS FINAIS ---
print("-" * 40)
print(f"🏆 Precisão RANDOM FOREST: {acc_rf * 100:.2f}%")
print(f"🚀 Precisão XGBOOST:      {acc_xgb * 100:.2f}%")
print("-" * 40)

A preparar os dados e a iniciar os testes no Laboratório...

----------------------------------------
🏆 Precisão RANDOM FOREST: 53.11%
🚀 Precisão XGBOOST:      54.10%
----------------------------------------
